In [ ]:
import sys, subprocess, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # Install cuML via RAPIDS pip index (requires CUDA 12 runtime on Colab)
    _pip('cuml-cu12', '--extra-index-url', 'https://pypi.nvidia.com')
    print('cuML installed.')
    # Mount Google Drive and set project root
    # Upload the entire eeg-ml-benchmark-suite folder to your Drive first
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_BASE_DIR = '/content/drive/MyDrive/eeg-ml-benchmark-suite'
    COLAB_MODEL_DIR = os.path.join(COLAB_BASE_DIR, 'Model_KNN')
    print(f'Project root: {COLAB_BASE_DIR}')
else:
    COLAB_BASE_DIR = None
    COLAB_MODEL_DIR = None
    print('Running locally for KNN.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time, sys, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score

# cuML drop-in replacement (GPU); falls back to sklearn on CPU
try:
    from cuml.neighbors import KNeighborsClassifier
    USING_CUML = True
    print('Backend: cuML (GPU)')
except ImportError:
    from sklearn.neighbors import KNeighborsClassifier
    USING_CUML = False
    print('Backend: sklearn (CPU)')

# Add model folder to sys.path so config.py is importable
_cfg_dir = COLAB_MODEL_DIR if (COLAB_MODEL_DIR and os.path.isdir(COLAB_MODEL_DIR)) \
           else (os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())
sys.path.insert(0, _cfg_dir)
import config


In [ ]:
MODEL_NAME = 'KNN'
TASK_LABEL = 'Binary'
FEATURES_LABEL = 'Raw'

# Resolve project root (Colab Drive path takes priority over local path)
if 'COLAB_BASE_DIR' in dir() and COLAB_BASE_DIR and os.path.isdir(COLAB_BASE_DIR):
    _base_dir = COLAB_BASE_DIR
else:
    _nb_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
    _base_dir = os.path.dirname(_nb_dir)
DATA_DIR = os.path.join(_base_dir, 'Data')
RESULTS_DIR = os.path.join(_base_dir, 'Results')
METRICS_DIR = os.path.join(RESULTS_DIR, 'KNN')
PLOTS_DIR = os.path.join(RESULTS_DIR, 'KNN', f'{TASK_LABEL}_{FEATURES_LABEL}')
GLOBAL_METRICS = os.path.join(RESULTS_DIR, 'all_models_metrics.csv')
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_bin_raw.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val_bin_raw.csv'))

X_train = train_df.drop(columns=['y']).values.astype(np.float32)
y_train = train_df['y'].values.astype(np.int32)
X_val = val_df.drop(columns=['y']).values.astype(np.float32)
y_val = val_df['y'].values.astype(np.int32)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Classes: {sorted(set(y_train.tolist()))}')


In [ ]:
from tqdm.auto import tqdm
import contextlib, joblib
from sklearn.model_selection import ParameterGrid

@contextlib.contextmanager
def tqdm_joblib(tqdm_obj):
    """Patch joblib to report progress into a tqdm bar."""
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_obj.update(n=self.batch_size)
            remaining = tqdm_obj.total - tqdm_obj.n
            tqdm_obj.set_postfix_str(f'{remaining} fits left')
            return super().__call__(*args, **kwargs)
    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _Cb
    try:
        yield tqdm_obj
    finally:
        joblib.parallel.BatchCompletionCallBack = old
        tqdm_obj.close()

estimator = KNeighborsClassifier()

param_grid = config.PARAM_GRID
n_jobs = 1 if USING_CUML else -1  # cuML manages GPU threads; sklearn uses CPU cores

if config.SEARCH_TYPE == 'grid':
    search = GridSearchCV(
        estimator, param_grid,
        cv=config.CV_FOLDS, scoring=config.SCORING,
        n_jobs=n_jobs, verbose=3,
    )
    total_fits = len(list(ParameterGrid(param_grid))) * config.CV_FOLDS
else:
    search = RandomizedSearchCV(
        estimator, param_grid,
        n_iter=config.N_ITER, cv=config.CV_FOLDS,
        scoring=config.SCORING, n_jobs=n_jobs,
        verbose=3, random_state=42,
    )
    total_fits = config.N_ITER * config.CV_FOLDS

t0 = time.time()
with tqdm_joblib(tqdm(total=total_fits, desc=f'{MODEL_NAME} CV', unit='fit')):
    search.fit(X_train, y_train)
search_time = time.time() - t0
print(f'Search finished in {search_time:.1f}s')
print(f'Best params: {search.best_params_}')
print(f'Best CV acc: {search.best_score_:.4f}')

# Save ALL parameter combinations to per-model CV CSV (upsert by Task+Features)
cv_df = pd.DataFrame(search.cv_results_)
cv_df['Task'] = TASK_LABEL
cv_df['Features'] = FEATURES_LABEL
cv_path = os.path.join(METRICS_DIR, config.CV_RESULTS_FILE)
if os.path.exists(cv_path):
    old = pd.read_csv(cv_path)
    old = old[~((old['Task'] == TASK_LABEL) & (old['Features'] == FEATURES_LABEL))]
    cv_df = pd.concat([old, cv_df], ignore_index=True)
cv_df.to_csv(cv_path, index=False)
print(f'CV results saved -> {cv_path}  ({len(cv_df)} rows total)')


In [ ]:
results_df = pd.DataFrame(search.cv_results_)

if config.PLOT_TYPE == 'bar':
    top_n = min(20, len(results_df))
    top_df = results_df.nlargest(top_n, 'mean_test_score').reset_index(drop=True)
    param_cols = [c for c in top_df.columns if c.startswith('param_')]
    labels = top_df[param_cols].astype(str).agg(' | '.join, axis=1)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(range(len(top_df)), top_df['mean_test_score'],
           yerr=top_df['std_test_score'], capsize=3, alpha=0.8, color='steelblue')
    ax.set_xticks(range(len(top_df)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Mean CV Accuracy')
    ax.set_title(f'{config.PLOT_TITLE} - Top {top_n} Combinations ({TASK_LABEL} / {FEATURES_LABEL})')
    ax.set_ylim(max(0, top_df['mean_test_score'].min() - 0.05), 1.0)
    plt.tight_layout()
    plt.show()

elif config.PLOT_TYPE == 'line':
    x_vals = results_df[config.PLOT_X_AXIS].astype(float)
    sort_idx = x_vals.argsort()
    xs = x_vals.iloc[sort_idx].values
    ys = results_df['mean_test_score'].iloc[sort_idx].values
    es = results_df['std_test_score'].iloc[sort_idx].values

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(xs, ys, marker='o', linewidth=2, color='steelblue')
    ax.fill_between(xs, ys - es, ys + es, alpha=0.3, color='steelblue')
    ax.set_xscale('log')
    ax.set_xlabel(config.PLOT_X_AXIS.replace('param_', ''))
    ax.set_ylabel('Mean CV Accuracy')
    ax.set_title(f'{config.PLOT_TITLE} - Parameter Search ({TASK_LABEL} / {FEATURES_LABEL})')
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

best_model = search.best_estimator_

t_start = time.time()
best_model.fit(X_train, y_train)
train_time = time.time() - t_start

y_pred_raw = best_model.predict(X_val)

# Convert cuML/cupy outputs to plain numpy int array
if USING_CUML:
    try:
        y_pred = np.asarray(y_pred_raw.values_host).astype(int)
    except AttributeError:
        try:
            y_pred = np.asarray(y_pred_raw.get()).astype(int)
        except AttributeError:
            y_pred = np.asarray(y_pred_raw).astype(int)
else:
    y_pred = np.asarray(y_pred_raw).astype(int)

acc = accuracy_score(y_val, y_pred)
f1  = f1_score(y_val, y_pred, average='weighted')
print(f'Validation Accuracy: {acc:.4f}')
print(f'Validation F1 (wtd): {f1:.4f}')
print(f'Train time: {train_time:.2f}s')

cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, colorbar=False)
ax.set_title(f'{MODEL_NAME} ({TASK_LABEL} / {FEATURES_LABEL})\nAcc={acc:.4f}  F1={f1:.4f}')
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, f'{MODEL_NAME}_{TASK_LABEL}_{FEATURES_LABEL}.png')
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved -> {plot_path}')


In [ ]:
metrics_file = GLOBAL_METRICS

record = {
    'Model': MODEL_NAME,
    'Task': TASK_LABEL,
    'Features': FEATURES_LABEL,
    'Best_Param': str(search.best_params_),
    'CV_Score': round(float(search.best_score_), 4),
    'Val_Acc': round(float(acc), 4),
    'Val_F1': round(float(f1), 4),
    'Train_Time': round(float(train_time), 2),
}

record_df = pd.DataFrame([record])
if os.path.exists(metrics_file):
    existing = pd.read_csv(metrics_file)
    mask = ~(
        (existing['Model'] == record['Model']) &
        (existing['Task'] == record['Task']) &
        (existing['Features'] == record['Features'])
    )
    updated = pd.concat([existing[mask], record_df], ignore_index=True)
    updated.to_csv(metrics_file, index=False)
else:
    record_df.to_csv(metrics_file, index=False)

print(f'Saved to {metrics_file}')
print(record_df.to_string(index=False))
